# End-to-End Fusion Fine-Tuning

Trains MentalRoBERTa backbone **and** the late-concat fusion head **jointly** in a single pass.

### Why this is different from the existing fine-tuning notebook

| Stage | Existing pipeline | This notebook |
|---|---|---|
| Stage 1 | Fine-tune RoBERTa with classification head | — |
| Stage 2 | Extract **frozen** CLS embeddings | — |
| Stage 3 | Train fusion MLP on frozen embeddings | — |
| **Single stage** | — | **Fine-tune backbone + fusion layers together** |

In the two-stage pipeline the fusion loss never reaches the transformer — the backbone is already frozen when the fusion layers train.  
Here, the gradient from confused pairs (Bipolar vs Depression) flows all the way back through the attention layers, nudging the encoder to produce representations that are more separable **given** the handcrafted signal.

### Two learning rates (critical)
- **Backbone** `2e-5` — small, preserves pre-trained knowledge  
- **Fusion layers** `1e-4` — 5× larger, fusion layers start from random init

### Initialisation
Starts from the already fine-tuned backbone saved at `results/models/roberta/finetuned/`.  
This gives a warm start — the backbone already knows the task domain — so 3–5 epochs are enough.

**Pipeline:**
1. Load processed CSVs + pre-extracted affective / handcrafted features from parquet
2. Fit StandardScaler on training handcrafted features
3. Build `EndToEndFusionModel` (backbone + fusion)
4. Train with differential LRs, early stopping on val macro-F1
5. Evaluate best checkpoint on test set
6. Save model + artifacts to Drive

**Expected inputs:**
- `data/processed/{train,val,test}.csv`
- `data/features/affective/{split}/{goemotions,vad,vader}.parquet`
- `data/features/{lexical,syntactic,structural}/{split}/*.parquet`
- `results/models/roberta/finetuned/` (from the fine-tuning notebook)

**Outputs:**
- Best checkpoint → `results/models/e2e/checkpoints/best.pt`
- Evaluation → `results/e2e/evaluation/`
- Training history → `results/e2e/logs/history.json`

## 1. Runtime and Dependencies

In [ ]:
!nvidia-smi

In [ ]:
!pip -q install transformers accelerate pandas pyarrow scikit-learn matplotlib seaborn tqdm

## 2. Mount Drive and Configure Paths

Set `PROJECT_DIR` to wherever `mental_health_fusion` lives in your Drive.  
Set `INIT_FROM` to `'finetuned'` (recommended) to warm-start from the already fine-tuned backbone,  
or `'pretrained'` to train from `mental/mental-roberta-base` from scratch.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

# ── CHANGE THIS ────────────────────────────────────────────────────────────
PROJECT_DIR = Path('/content/drive/MyDrive/mental_health_fusion')

# 'finetuned' = warm-start from the backbone saved by colab_finetune_mental_roberta_and_embeddings_extraction.ipynb
# 'pretrained' = start from mental/mental-roberta-base (slower, more epochs needed)
INIT_FROM = 'finetuned'
# ──────────────────────────────────────────────────────────────────────────

PROCESSED_DIR    = PROJECT_DIR / 'data' / 'processed'
FEATURES_DIR     = PROJECT_DIR / 'data' / 'features'
FINETUNED_DIR    = PROJECT_DIR / 'results' / 'models' / 'roberta' / 'finetuned'
E2E_CKPT_DIR     = PROJECT_DIR / 'results' / 'models' / 'e2e' / 'checkpoints'
E2E_EVAL_DIR     = PROJECT_DIR / 'results' / 'e2e' / 'evaluation'
E2E_LOG_DIR      = PROJECT_DIR / 'results' / 'e2e' / 'logs'

for d in [E2E_CKPT_DIR, E2E_EVAL_DIR, E2E_LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Hyperparameters ────────────────────────────────────────────────────────
PRETRAINED_MODEL  = 'mental/mental-roberta-base'
MAX_LENGTH        = 512
BATCH_SIZE        = 16       # fits T4 (15 GB); use 8 if OOM
BACKBONE_LR       = 2e-5     # small LR — backbone is already partially tuned
FUSION_LR         = 1e-4     # 5× larger — fusion layers start from random init
NUM_EPOCHS        = 5
WEIGHT_DECAY      = 0.01
WARMUP_RATIO      = 0.1
GRAD_CLIP         = 1.0
LABEL_SMOOTHING   = 0.1      # reduces overconfidence on ambiguous pairs
SEED              = 42
NUM_LABELS        = 6

# Feature dimensions — must match what was extracted
AFFECTIVE_DIM    = 34   # goemotions(28) + vad(3) + vader(3)
HANDCRAFTED_DIM  = 26   # lexical(11) + syntactic(8) + structural(7)
SEMANTIC_DIM     = 768

# Projection dims — same as LateConcatFusion
SEMANTIC_PROJ    = 256
AFFECTIVE_PROJ   = 128
HANDCRAFTED_PROJ = 64

ID_TO_CLASS = {0: 'ADHD', 1: 'Anxiety', 2: 'Bipolar', 3: 'Depression', 4: 'PTSD', 5: 'None'}

backbone_source = str(FINETUNED_DIR) if (INIT_FROM == 'finetuned' and FINETUNED_DIR.exists()) else PRETRAINED_MODEL
print(f'Backbone initialised from : {backbone_source}')
print(f'Batch size                : {BATCH_SIZE}')
print(f'Backbone LR / Fusion LR   : {BACKBONE_LR} / {FUSION_LR}')
print(f'Epochs                    : {NUM_EPOCHS}')

## 3. Imports and Reproducibility

In [ ]:
import json
import random
import time

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix,
)
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from tqdm.auto import tqdm


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 4. Load Processed CSVs

In [ ]:
train_df = pd.read_csv(PROCESSED_DIR / 'train.csv')
val_df   = pd.read_csv(PROCESSED_DIR / 'val.csv')
test_df  = pd.read_csv(PROCESSED_DIR / 'test.csv')

for name, df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    dist = df['class_id'].value_counts().sort_index().rename(ID_TO_CLASS)
    print(f'{name} ({len(df)} rows):')
    print(dist.to_string())
    print()

## 5. Load Pre-extracted Features

Affective and handcrafted features are pre-extracted and fixed — they do **not** update during training.  
Only the RoBERTa backbone (semantic branch) is trainable.  
Handcrafted features are StandardScaler-normalised (fit on train, transform val/test).

In [ ]:
def load_feature_group(base_dir, split, sub_names):
    """Load sub-feature parquets for one split and concatenate horizontally."""
    matrices = []
    for sub_name in sub_names:
        path = base_dir / split / f'{sub_name}.parquet'
        df = pd.read_parquet(path)
        if 'features' in df.columns:
            mat = np.array(df['features'].tolist(), dtype=np.float32)
        else:
            # Wide format (numeric columns only)
            mat = df.select_dtypes(include=[np.number]).values.astype(np.float32)
        matrices.append(mat)
    combined = np.concatenate(matrices, axis=1)
    print(f'  {split} {[s for s in sub_names]} -> {combined.shape}')
    return combined


AFFECTIVE_SUBS   = ['goemotions', 'vad', 'vader']
LEXICAL_SUBS     = ['diversity', 'word_rates', 'pronouns', 'punctuation']
SYNTACTIC_SUBS   = ['complexity', 'pos_ratios', 'readability']
STRUCTURAL_SUBS  = ['coherence', 'tense']

print('Loading affective features...')
af_tr = load_feature_group(FEATURES_DIR / 'affective', 'train', AFFECTIVE_SUBS)
af_va = load_feature_group(FEATURES_DIR / 'affective', 'val',   AFFECTIVE_SUBS)
af_te = load_feature_group(FEATURES_DIR / 'affective', 'test',  AFFECTIVE_SUBS)

print('\nLoading handcrafted features (lexical + syntactic + structural)...')
def load_handcrafted(split):
    return np.concatenate([
        load_feature_group(FEATURES_DIR / 'lexical',    split, LEXICAL_SUBS),
        load_feature_group(FEATURES_DIR / 'syntactic',  split, SYNTACTIC_SUBS),
        load_feature_group(FEATURES_DIR / 'structural', split, STRUCTURAL_SUBS),
    ], axis=1)

hc_tr = load_handcrafted('train')
hc_va = load_handcrafted('val')
hc_te = load_handcrafted('test')

print(f'\nHandcrafted (raw)  train={hc_tr.shape}  val={hc_va.shape}  test={hc_te.shape}')
print(f'Affective          train={af_tr.shape}  val={af_va.shape}  test={af_te.shape}')

# Validate dimensions match config
assert hc_tr.shape[1] == HANDCRAFTED_DIM, f'Expected {HANDCRAFTED_DIM}, got {hc_tr.shape[1]}'
assert af_tr.shape[1] == AFFECTIVE_DIM,   f'Expected {AFFECTIVE_DIM}, got {af_tr.shape[1]}'

# Scale handcrafted features
scaler = StandardScaler()
hc_tr = scaler.fit_transform(hc_tr).astype(np.float32)
hc_va = scaler.transform(hc_va).astype(np.float32)
hc_te = scaler.transform(hc_te).astype(np.float32)
joblib.dump(scaler, E2E_CKPT_DIR / 'handcrafted_scaler.joblib')
print('\nHandcrafted features scaled and scaler saved.')

## 6. Model Definition

**Architecture** — identical projection/classifier structure to `LateConcatFusion` but the semantic branch is a live trainable backbone instead of a frozen embedding lookup:

```
text  ──► RoBERTa encoder (trainable, LR=2e-5) ──► CLS (768)
                                                         ──► Linear(768→256) ──► LayerNorm ──► GELU ──► Dropout ──┐
affective(34)   ──► Linear(34→128)  ──► LayerNorm ──► GELU ──► Dropout ──────────────────────────────────────────┤
handcrafted(26) ──► Linear(26→64)   ──► LayerNorm ──► GELU ──► Dropout ──────────────────────────────────────────┤
                                                                                                   concat(448) ──► ClassifierHead ──► 6 logits
```

In [ ]:
class ProjectionBlock(nn.Module):
    def __init__(self, in_dim, out_dim, dropout=0.1):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim)
        self.norm   = nn.LayerNorm(out_dim)
        self.drop   = nn.Dropout(dropout)

    def forward(self, x):
        return self.drop(F.gelu(self.norm(self.linear(x))))


class ClassifierHead(nn.Module):
    def __init__(self, in_dim, hidden=256, num_labels=6, dropout=0.3):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, num_labels),
        )

    def forward(self, x):
        return self.layers(x)


class EndToEndFusionModel(nn.Module):
    """
    Trainable RoBERTa backbone + late-concat fusion head.

    Pass backbone_parameters() and fusion_parameters() to separate
    AdamW parameter groups with different learning rates.
    """

    def __init__(self, backbone_path, affective_dim=AFFECTIVE_DIM,
                 handcrafted_dim=HANDCRAFTED_DIM, num_labels=NUM_LABELS, dropout=0.1):
        super().__init__()
        self.backbone         = AutoModel.from_pretrained(backbone_path)
        self.semantic_proj    = ProjectionBlock(SEMANTIC_DIM,     SEMANTIC_PROJ,    dropout)
        self.affective_proj   = ProjectionBlock(affective_dim,    AFFECTIVE_PROJ,   dropout)
        self.handcrafted_proj = ProjectionBlock(handcrafted_dim,  HANDCRAFTED_PROJ, dropout)
        fusion_dim            = SEMANTIC_PROJ + AFFECTIVE_PROJ + HANDCRAFTED_PROJ  # 448
        self.classifier       = ClassifierHead(fusion_dim, hidden=256, num_labels=num_labels)

    def backbone_parameters(self):
        return list(self.backbone.parameters())

    def fusion_parameters(self):
        return (
            list(self.semantic_proj.parameters())
            + list(self.affective_proj.parameters())
            + list(self.handcrafted_proj.parameters())
            + list(self.classifier.parameters())
        )

    def forward(self, input_ids, attention_mask, affective, handcrafted):
        cls = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
        ).last_hidden_state[:, 0]          # (B, 768)

        fused = torch.cat([
            self.semantic_proj(cls),
            self.affective_proj(affective),
            self.handcrafted_proj(handcrafted),
        ], dim=1)                          # (B, 448)
        return self.classifier(fused)

## 7. Dataset and DataLoaders

In [ ]:
class E2EDataset(Dataset):
    """Returns tokenised text + pre-extracted feature tensors + label."""

    def __init__(self, df, tokenizer, max_length, affective, handcrafted):
        self.texts      = df['text'].fillna('').astype(str).tolist()
        self.labels     = df['class_id'].tolist()
        self.tok        = tokenizer
        self.maxlen     = max_length
        self.affective  = torch.from_numpy(affective)
        self.handcrafted = torch.from_numpy(handcrafted)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tok(
            self.texts[idx],
            max_length=self.maxlen,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'affective':      self.affective[idx],
            'handcrafted':    self.handcrafted[idx],
            'labels':         torch.tensor(self.labels[idx], dtype=torch.long),
        }

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(backbone_source)

def make_loader(df, aff, hc, shuffle):
    return DataLoader(
        E2EDataset(df, tokenizer, MAX_LENGTH, aff, hc),
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=0,
        pin_memory=True,
    )

train_loader = make_loader(train_df, af_tr, hc_tr, shuffle=True)
val_loader   = make_loader(val_df,   af_va, hc_va, shuffle=False)
test_loader  = make_loader(test_df,  af_te, hc_te, shuffle=False)

print(f'Train batches : {len(train_loader)}')
print(f'Val   batches : {len(val_loader)}')
print(f'Test  batches : {len(test_loader)}')

## 8. Model, Optimizer, and Scheduler

Two AdamW parameter groups:
- `backbone_parameters()` → `BACKBONE_LR = 2e-5`  
- `fusion_parameters()` → `FUSION_LR = 1e-4`

Label smoothing (`0.1`) penalises overconfident predictions on ambiguous pairs like Bipolar/Depression.

In [ ]:
model = EndToEndFusionModel(backbone_path=backbone_source).to(device)

backbone_params = sum(p.numel() for p in model.backbone_parameters())
fusion_params   = sum(p.numel() for p in model.fusion_parameters())
print(f'Backbone params : {backbone_params:,}  (LR={BACKBONE_LR})')
print(f'Fusion params   : {fusion_params:,}  (LR={FUSION_LR})')
print(f'Total trainable : {backbone_params + fusion_params:,}')

optimizer = torch.optim.AdamW(
    [
        {'params': model.backbone_parameters(), 'lr': BACKBONE_LR},
        {'params': model.fusion_parameters(),   'lr': FUSION_LR},
    ],
    weight_decay=WEIGHT_DECAY,
)

total_steps  = len(train_loader) * NUM_EPOCHS
warmup_steps = int(WARMUP_RATIO * total_steps)
scheduler    = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

print(f'\nTotal steps  : {total_steps}')
print(f'Warmup steps : {warmup_steps}')

## 9. Training and Evaluation Helpers

In [ ]:
def train_epoch(model, loader, optimizer, scheduler, criterion, device):
    model.train()
    total_loss, all_preds, all_labels = 0.0, [], []

    for batch in tqdm(loader, desc='  train', leave=False):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        affective      = batch['affective'].to(device)
        handcrafted    = batch['handcrafted'].to(device)
        labels         = batch['labels'].to(device)

        optimizer.zero_grad()
        logits = model(input_ids, attention_mask, affective, handcrafted)
        loss   = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        all_preds.extend(logits.argmax(dim=-1).cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

    avg_loss = total_loss / len(loader)
    acc = accuracy_score(all_labels, all_preds)
    f1  = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return avg_loss, acc, f1


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []

    for batch in tqdm(loader, desc='  eval ', leave=False):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        affective      = batch['affective'].to(device)
        handcrafted    = batch['handcrafted'].to(device)
        labels         = batch['labels'].to(device)

        logits = model(input_ids, attention_mask, affective, handcrafted)
        total_loss += criterion(logits, labels).item()
        all_preds.extend(logits.argmax(dim=-1).cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

    avg_loss = total_loss / len(loader)
    acc = accuracy_score(all_labels, all_preds)
    f1  = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return avg_loss, acc, f1, all_preds, all_labels

## 10. Training Loop

Best checkpoint is saved by **validation macro-F1** (more robust than accuracy for imbalanced classes).

In [ ]:
history = {
    'train_loss': [], 'train_acc': [], 'train_f1': [],
    'val_loss':   [], 'val_acc':   [], 'val_f1':   [],
}

best_val_f1  = 0.0
best_state   = None
best_epoch   = 0

for epoch in range(1, NUM_EPOCHS + 1):
    print(f'\n{"="*60}')
    print(f'Epoch {epoch}/{NUM_EPOCHS}')
    print(f'{"="*60}')
    t0 = time.time()

    tr_loss, tr_acc, tr_f1 = train_epoch(model, train_loader, optimizer, scheduler, criterion, device)
    va_loss, va_acc, va_f1, _, _ = evaluate(model, val_loader, criterion, device)

    history['train_loss'].append(tr_loss)
    history['train_acc'].append(tr_acc)
    history['train_f1'].append(tr_f1)
    history['val_loss'].append(va_loss)
    history['val_acc'].append(va_acc)
    history['val_f1'].append(va_f1)

    print(f'  Train  loss={tr_loss:.4f}  acc={tr_acc:.4f}  macro-F1={tr_f1:.4f}  ({time.time()-t0:.0f}s)')
    print(f'  Val    loss={va_loss:.4f}  acc={va_acc:.4f}  macro-F1={va_f1:.4f}')

    if va_f1 > best_val_f1:
        best_val_f1 = va_f1
        best_epoch  = epoch
        best_state  = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        torch.save(best_state, E2E_CKPT_DIR / 'best.pt')
        print(f'  -> Best checkpoint saved  (val macro-F1={best_val_f1:.4f})')

print(f'\nBest val macro-F1 : {best_val_f1:.4f}  (epoch {best_epoch})')

## 11. Training Curves

In [ ]:
epochs_range = range(1, len(history['train_loss']) + 1)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

labels_map = {'loss': 'Cross-Entropy Loss', 'acc': 'Accuracy', 'f1': 'Macro F1'}
for ax, metric in zip(axes, ['loss', 'acc', 'f1']):
    ax.plot(epochs_range, history[f'train_{metric}'], marker='o', label='Train')
    ax.plot(epochs_range, history[f'val_{metric}'],   marker='s', label='Val')
    ax.set_title(labels_map[metric])
    ax.set_xlabel('Epoch')
    ax.legend()
    ax.grid(alpha=0.3)

plt.suptitle('End-to-End Fusion — Training Curves', y=1.02)
plt.tight_layout()
plt.savefig(str(E2E_EVAL_DIR / 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

## 12. Evaluate Best Checkpoint on Test Set

In [ ]:
# Reload best weights
model.load_state_dict(best_state)

te_loss, te_acc, te_f1, te_preds, te_labels = evaluate(model, test_loader, criterion, device)

print(f'Test results  (best checkpoint — val macro-F1={best_val_f1:.4f}  epoch={best_epoch})')
print(f'  Loss      : {te_loss:.4f}')
print(f'  Accuracy  : {te_acc:.4f}  ({te_acc*100:.2f}%)')
print(f'  Macro-F1  : {te_f1:.4f}')

## 13. Per-Class Classification Report

In [ ]:
class_names = [ID_TO_CLASS[i] for i in range(NUM_LABELS)]

report_str  = classification_report(te_labels, te_preds, target_names=class_names, digits=4)
report_dict = classification_report(te_labels, te_preds, target_names=class_names,
                                    digits=4, output_dict=True)
print(report_str)

report_df = pd.DataFrame(report_dict).T
report_df.to_csv(E2E_EVAL_DIR / 'classification_report.csv')
display(report_df.round(4))

## 14. Confusion Matrix

In [ ]:
cm      = confusion_matrix(te_labels, te_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, data, fmt, title in [
    (axes[0], cm,      'd',    'Counts'),
    (axes[1], cm_norm, '.2f',  'Row-normalised'),
]:
    sns.heatmap(data, annot=True, fmt=fmt, cmap='Blues',
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_title(f'Confusion Matrix ({title})')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')

plt.tight_layout()
plt.savefig(str(E2E_EVAL_DIR / 'confusion_matrix.png'), dpi=150)
plt.show()

## 15. Save Summary JSON

In [ ]:
per_class_acc = {}
te_labels_arr = np.array(te_labels)
te_preds_arr  = np.array(te_preds)
for cls_id, cls_name in ID_TO_CLASS.items():
    mask = te_labels_arr == cls_id
    per_class_acc[cls_name] = round(float((te_preds_arr[mask] == cls_id).mean()), 4)

summary = {
    'backbone_source'   : backbone_source,
    'backbone_lr'       : BACKBONE_LR,
    'fusion_lr'         : FUSION_LR,
    'num_epochs'        : NUM_EPOCHS,
    'batch_size'        : BATCH_SIZE,
    'label_smoothing'   : LABEL_SMOOTHING,
    'best_epoch'        : best_epoch,
    'best_val_macro_f1' : round(best_val_f1, 6),
    'test_accuracy'     : round(te_acc, 6),
    'test_macro_f1'     : round(te_f1, 6),
    'test_loss'         : round(te_loss, 6),
    'per_class_acc'     : per_class_acc,
    'history'           : history,
}

with open(E2E_EVAL_DIR / 'summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
with open(E2E_LOG_DIR / 'history.json', 'w') as f:
    json.dump(history, f, indent=2)

print('Summary saved to :', E2E_EVAL_DIR / 'summary.json')
print('History saved to :', E2E_LOG_DIR / 'history.json')
print('Checkpoint at    :', E2E_CKPT_DIR / 'best.pt')
print()
print('=== Final Results ===')
print(f'  Val  macro-F1 : {best_val_f1:.4f}')
print(f'  Test accuracy : {te_acc:.4f}  ({te_acc*100:.2f}%)')
print(f'  Test macro-F1 : {te_f1:.4f}')
print()
for name, acc in per_class_acc.items():
    print(f'  {name:<12} {acc:.4f}')

## 16. (Optional) Continue Training

If val macro-F1 was still improving at the last epoch, run this cell to train for additional epochs.  
After running, re-execute sections 12–15 to update evaluation and artifacts.

In [ ]:
ADDITIONAL_EPOCHS = 2  # adjust as needed

# Re-init optimizer and scheduler for the additional steps
cont_optimizer = torch.optim.AdamW(
    [
        {'params': model.backbone_parameters(), 'lr': BACKBONE_LR},
        {'params': model.fusion_parameters(),   'lr': FUSION_LR},
    ],
    weight_decay=WEIGHT_DECAY,
)
cont_total_steps  = len(train_loader) * ADDITIONAL_EPOCHS
cont_warmup_steps = int(WARMUP_RATIO * cont_total_steps)
cont_scheduler    = get_linear_schedule_with_warmup(
    cont_optimizer, num_warmup_steps=cont_warmup_steps,
    num_training_steps=cont_total_steps,
)

for epoch_offset in range(1, ADDITIONAL_EPOCHS + 1):
    current_epoch = NUM_EPOCHS + epoch_offset
    print(f'\n{"="*60}')
    print(f'Epoch {current_epoch}/{NUM_EPOCHS + ADDITIONAL_EPOCHS} (continued)')
    print(f'{"="*60}')

    tr_loss, tr_acc, tr_f1 = train_epoch(model, train_loader, cont_optimizer, cont_scheduler, criterion, device)
    va_loss, va_acc, va_f1, _, _ = evaluate(model, val_loader, criterion, device)

    history['train_loss'].append(tr_loss)
    history['train_acc'].append(tr_acc)
    history['train_f1'].append(tr_f1)
    history['val_loss'].append(va_loss)
    history['val_acc'].append(va_acc)
    history['val_f1'].append(va_f1)

    print(f'  Train  loss={tr_loss:.4f}  acc={tr_acc:.4f}  macro-F1={tr_f1:.4f}')
    print(f'  Val    loss={va_loss:.4f}  acc={va_acc:.4f}  macro-F1={va_f1:.4f}')

    if va_f1 > best_val_f1:
        best_val_f1 = va_f1
        best_epoch  = current_epoch
        best_state  = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        torch.save(best_state, E2E_CKPT_DIR / 'best.pt')
        print(f'  -> New best checkpoint (val macro-F1={best_val_f1:.4f})')

# Re-run sections 12-15 after this cell to update evaluation artifacts